# Phân khúc khách hàng Online Retail
Notebook trình bày kết quả của pipeline tích hợp. Chạy `python src/pipeline.py` từ thư mục gốc trước khi mở; báo cáo đầy đủ ở `docs/REPORT.md`.
Các mô hình dùng cùng khách hàng và cùng ma trận RFM; notebook không fit lại mô hình theo cấu hình khác.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
OUT = ROOT / "outputs"
summary = json.loads((OUT / "run_summary.json").read_text())
audit = json.loads((OUT / "data_audit.json").read_text())
print("Customers:", summary["customer_count"])
print("Primary model:", summary["primary_model"])
print("Preprocessing:", summary["preprocessing"])
pd.DataFrame(audit["counts"].items(), columns=["Stage", "Count"])

## Phân phối và tiền xử lý
RFM lệch phải: biến đổi log1p rồi chuẩn hóa. Chi tiêu ròng trong kỳ chưa phải lợi nhuận và chưa đối soát từng dòng hoàn với đơn mua.

In [ ]:
def show(name):
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(mpimg.imread(OUT / "figures" / name))
    ax.axis("off")
    plt.show()
show("rfm_distributions.png")

## Chọn số nhóm và so sánh
K-Means, GMM và Ward cùng k=4 là phép đối chiếu trực tiếp. RFM baseline và GMM BIC có số nhóm khác, cần đọc riêng về độ chi tiết.

In [ ]:
show("kmeans_selection.png")
pd.read_csv(OUT / "tables/model_comparison.csv")

## Độ ổn định và độ nhạy
ARI đo mức trùng khớp nhãn giữa các seed, không phải accuracy. Metrics ở các ma trận chặn/không chặn không được gộp làm một bảng xếp hạng.

In [ ]:
stability = pd.read_csv(OUT / "tables/seed_stability.csv")
print(stability.groupby("Model").ARI.agg(["min", "median"]))
print(pd.read_csv(OUT / "tables/preprocessing_sensitivity.csv").to_string(index=False))
pd.read_csv(OUT / "gmm_bic/tables/regularization_sensitivity.csv")

## Chân dung và đề xuất
Tên nhóm là mô tả tương đối. Các đề xuất là giả thuyết cần A/B test và kiểm soát chi phí. GMM_Confidence không phải xác suất nhãn marketing đúng.

In [ ]:
show("kmeans_pca.png")
show("customer_value_shares.png")
pd.read_csv(OUT / "tables/segment_profiles.csv")

In [ ]:
pd.read_csv(OUT / "tables/segment_strategies.csv")

## Giới hạn và tái lập
Dữ liệu lịch sử của một nhà bán lẻ, không có ground truth hay kiểm chứng hiệu quả marketing. Cần kiểm tra theo thời gian trước khi triển khai. Xem `docs/REPORT.md`, `outputs/data_audit.json` và phiên bản thư viện trong `outputs/run_summary.json`. Chạy `python src/verify_project.py` để đối chiếu dữ liệu, model, probabilities, metrics và profiles đã lưu.